# Pipeline Corpus V6 — SME+Patch · Clause-Dense · BM25(k1=1.5,b=0.4) · Adaptive Cutoff

**Pipeline:**
1. **Corpus** = `corpus_luat_sme_merged_v3.jsonl` + `patch_laws_articles.jsonl` (~5,931 điều)
2. **BM25** (underthesea word-tok + 4-char n-gram, k1=1.5, b=0.4) → top-100 articles
3. **Dense BGE-M3** encode CLAUSE-level (sme_clauses_v4 + patch_clauses) → top-200 clauses → MaxRank → top-50 articles
4. **RRF** (k=60) fuse BM25 + clause-dense + enrichment legs → pool 50
5. **BGE-reranker-v2-m3** Sliding Window MaxP (1200-char chunks, overlap 200, MaxP per article)
6. **Adaptive Soft-Floor cutoff** (ABS_FLOOR=0.38, MAX_K=7, MIN_K=1)

**Files cần upload vào Kaggle dataset:**
- `corpus_luat_sme_merged_v3.jsonl` — SME corpus gốc
- `patch_laws_articles.jsonl` — 5 luật mới 2024-2025 (273 articles)
- `sme_clauses_v4.jsonl` — clause chunks SME corpus
- `patch_laws_clauses.jsonl` — clause chunks patch laws
- `R2AIStage1DATA.json` — bộ câu hỏi test
- `r2ai_enrichment.jsonl` *(optional)* — enrichment cache

In [13]:
# Cell 1 — Cài packages
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q sentence-transformers rank_bm25 underthesea

In [14]:
# Cell 2 — Imports + GPU setup
import json, gc, pickle, hashlib
from collections import defaultdict
import numpy as np
import torch
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from underthesea import word_tokenize

n_gpu = torch.cuda.device_count()
print(f"[*] GPUs: {n_gpu}")
EMBED_DEVICE  = "cuda:0" if n_gpu >= 1 else "cpu"
RERANK_DEVICE = "cuda:1" if n_gpu >= 2 else EMBED_DEVICE
print(f"[*] Embedder → {EMBED_DEVICE} | Reranker → {RERANK_DEVICE}")

[*] GPUs: 2
[*] Embedder → cuda:0 | Reranker → cuda:1


In [15]:
# Cell 3 — Config
DATASET_DIR      = "/kaggle/input/datasets/letuano5/r2ai-corpus"
CORPUS_PATH      = f"{DATASET_DIR}/corpus_luat_sme_merged_v3.jsonl"
PATCH_ART_PATH   = f"{DATASET_DIR}/patch_laws_articles.jsonl"
SME_CLAUSES_PATH = f"{DATASET_DIR}/sme_clauses_v4.jsonl"
PATCH_CLS_PATH   = f"{DATASET_DIR}/patch_laws_clauses.jsonl"
TESTSET_PATH     = f"{DATASET_DIR}/R2AIStage1DATA.json"
ENRICHMENT_PATH  = f"{DATASET_DIR}/r2ai_enrichment.jsonl"
OUTPUT_PATH      = "/kaggle/working/retrieval_v6.json"
CACHE_DIR        = "/kaggle/working/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# --- BM25 (tuned cho legal text) ---
BM25_K1   = 1.5    # default 1.2 → tăng: văn bản pháp luật lặp từ có chủ đích
BM25_B    = 0.4    # default 0.75 → giảm: ít phạt điều dài
BM25_K    = 100

# --- Dense (clause-level) ---
DENSE_CLAUSE_K = 200   # top clauses retrieved
DENSE_K        = 50    # top articles sau khi group clauses → articles
DENSE_BATCH    = 64
DENSE_MAX_LEN  = 256   # clause ngắn, 256 đủ

# --- RRF ---
RRF_K       = 60
CANDIDATE_K = 50

# --- Reranker ---
RERANK_BATCH = 64

# --- Adaptive soft-floor cutoff ---
ABS_FLOOR = 0.38   # giảm nhẹ từ 0.40 → bắt thêm recall
MAX_K     = 7      # tăng từ 6
MIN_K     = 1      # giảm từ 2 → không ép trả 2 nếu chỉ 1 cái đúng

# --- Query enrichment ---
USE_ENRICHMENT = True
USE_KEYWORDS   = True
USE_DECOMP     = True
USE_HYDE       = False

print("Config loaded.")
for p in [CORPUS_PATH, PATCH_ART_PATH, SME_CLAUSES_PATH, PATCH_CLS_PATH, TESTSET_PATH]:
    print(f"  {'✓' if os.path.exists(p) else '✗'} {os.path.basename(p)}")

Config loaded.
  ✓ corpus_luat_sme_merged_v3.jsonl
  ✓ patch_laws_articles.jsonl
  ✓ sme_clauses_v4.jsonl
  ✓ patch_laws_clauses.jsonl
  ✓ R2AIStage1DATA.json


In [16]:
# Cell 4 — Load corpus (SME v3 + patch) + clause lookup

def _iter_jsonl(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

# ---- Corpus ----
print("[*] Load corpus (SME + patch)...")
raw_articles = []
for path in [CORPUS_PATH, PATCH_ART_PATH]:
    if os.path.exists(path):
        for a in _iter_jsonl(path):
            raw_articles.append(a)
    else:
        print(f"  [!] Missing: {path}")

# Dedup theo submission_article_str, giữ bản full_text dài nhất
best: dict = {}
for a in raw_articles:
    meta = a.get("submission_article_str") or a.get("article_id", "")
    if not meta:
        continue
    body = a.get("full_text", "")
    if meta not in best or len(body) > len(best[meta].get("full_text", "")):
        best[meta] = a

metas    = list(best.keys())                                      # submission_article_str
docs     = [f"{best[m].get('article_title','')}".strip() + "\n" +
            best[m].get("full_text", "") for m in metas]          # BM25 + reranker text
doc_strs = [best[m].get("submission_doc_str", "") for m in metas] # cho output
meta_to_idx = {m: i for i, m in enumerate(metas)}

sig = hashlib.md5("|".join(metas).encode()).hexdigest()[:10]
print(f"  → {len(metas)} điều | {len(set(m.split('|')[0] for m in metas))} mã luật | sig={sig}")

# ---- Clause chunks (cho dense index) ----
print("[*] Load clause chunks cho dense index...")
print("[*] Load clause lookup...")
clause_list_meta: list = []   # (submission_article_str, text) cho dense index
n_clauses = 0
for path in [SME_CLAUSES_PATH, PATCH_CLS_PATH]:
    if not os.path.exists(path):
        print(f"  [!] Missing clause file: {path}")
        continue
    for c in _iter_jsonl(path):
        art_meta = c.get("submission_article_str") or c.get("article_id", "")
        text = c.get("text", "")
        if art_meta and text and art_meta in meta_to_idx:
            clause_list_meta.append((art_meta, text))
            n_clauses += 1

print(f"  → {n_clauses:,} clause chunks | {len(set(m for m,_ in clause_list_meta)):,} articles có clauses")
# print(f"  Articles không có clause: {len(missing_clauses)}" +
#       (f" — first: {missing_clauses[:2]}" if missing_clauses else " ✓"))

[*] Load corpus (SME + patch)...
  → 6151 điều | 62 mã luật | sig=da931588b8
[*] Load clause chunks cho dense index...
[*] Load clause lookup...
  → 41,746 clause chunks | 6,147 articles có clauses


In [17]:
# Cell 5 — Build BM25 (k1=1.5, b=0.4 | underthesea + 4-char n-gram)
bm25_cache = f"{CACHE_DIR}/bm25_{sig}_k{BM25_K1}_b{BM25_B}.pkl"


def hybrid_vi_tok(text: str) -> list:
    text = text.lower()
    try:
        words = word_tokenize(text, format="text").split()
    except Exception:
        words = text.split()
    ngrams = [text[i:i+4] for i in range(max(1, len(text) - 3))]
    return words + ngrams


if os.path.exists(bm25_cache):
    print("[*] Load BM25 tokenized corpus từ cache...")
    with open(bm25_cache, "rb") as f:
        tokenized_corpus = pickle.load(f)
else:
    print("[*] Tokenize corpus (underthesea + 4-gram)...")
    tokenized_corpus = [hybrid_vi_tok(d) for d in tqdm(docs, desc="Tok")]
    with open(bm25_cache, "wb") as f:
        pickle.dump(tokenized_corpus, f)

print(f"[*] Build BM25Okapi (k1={BM25_K1}, b={BM25_B})...")
bm25 = BM25Okapi(tokenized_corpus, k1=BM25_K1, b=BM25_B)
print("    BM25 ready.")

[*] Tokenize corpus (underthesea + 4-gram)...


Tok:   0%|          | 0/6151 [00:00<?, ?it/s]

[*] Build BM25Okapi (k1=1.5, b=0.4)...
    BM25 ready.


In [18]:
# Cell 6 — Build clause-level dense index (BGE-M3 fp16)
#
# Encode từng clause (≤256 tokens) thay vì full article.
# Vector clause tập trung hơn → dense retrieval chính xác hơn.
# Sau khi lấy top-K clauses → map về article index (MaxRank).

clause_texts = [t for _, t in clause_list_meta]
clause_art_idxs = np.array(
    [meta_to_idx[m] for m, _ in clause_list_meta], dtype=np.int32
)

clause_emb_cache = f"{CACHE_DIR}/clause_emb_{sig}.npy"

print("[*] Load BGE-M3...")
embedder = SentenceTransformer("BAAI/bge-m3", device=EMBED_DEVICE)
embedder.max_seq_length = DENSE_MAX_LEN
embedder.half()
torch.cuda.empty_cache()

if os.path.exists(clause_emb_cache):
    print("[*] Load clause embeddings từ cache...")
    clause_emb = np.load(clause_emb_cache)
else:
    print(f"[*] Encode {len(clause_texts):,} clauses (batch={DENSE_BATCH})...")
    clause_emb = embedder.encode(
        clause_texts,
        batch_size=DENSE_BATCH,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    np.save(clause_emb_cache, clause_emb)
    gc.collect(); torch.cuda.empty_cache()

print(f"    clause_emb: {clause_emb.shape}")


def clause_dense_retrieve(q_vec: np.ndarray, top_k_articles: int = DENSE_K) -> list:
    """Retrieve top_k_articles bằng cách tìm top clauses rồi group theo article (MaxRank)."""
    sims = clause_emb @ q_vec                                    # (N_clauses,)
    top_clause_idx = np.argsort(-sims)[:DENSE_CLAUSE_K]         # top-200 clauses
    seen_arts: dict = {}
    result: list = []
    for cidx in top_clause_idx.tolist():
        aidx = int(clause_art_idxs[cidx])
        if aidx not in seen_arts:
            seen_arts[aidx] = len(result)
            result.append(aidx)
            if len(result) >= top_k_articles:
                break
    return result

[*] Load BGE-M3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

[*] Encode 41,746 clauses (batch=64)...


Batches:   0%|          | 0/653 [00:00<?, ?it/s]

    clause_emb: (41746, 1024)


In [19]:
# Cell 7 — Load queries + encode query embeddings
print("[*] Load testset...")
with open(TESTSET_PATH, encoding="utf-8") as f:
    test_data = json.load(f)
print(f"    {len(test_data)} câu hỏi")

print("[*] Encode queries dense...")
q_texts = [it["question"] for it in test_data]
q_embs = embedder.encode(
    q_texts,
    batch_size=DENSE_BATCH,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)
q_emb_by_id = {it["id"]: q_embs[i] for i, it in enumerate(test_data)}
print(f"    Done. Shape: {q_embs.shape}")

[*] Load testset...
    2000 câu hỏi
[*] Encode queries dense...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

    Done. Shape: (2000, 1024)


In [20]:
# Cell 8 — Load enrichment cache + encode extra dense vectors
enrich_cache: dict = {}

if USE_ENRICHMENT and os.path.exists(ENRICHMENT_PATH):
    print(f"[*] Load enrichment...")
    with open(ENRICHMENT_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                d = json.loads(line)
                enrich_cache[d["id"]] = d
            except Exception:
                pass
    print(f"    {len(enrich_cache)} entries")
    for e in enrich_cache.values():
        kw_text = " ".join(e.get("keywords", []))
        e["keywords_tokens"]  = hybrid_vi_tok(kw_text) if kw_text else []
        e["sub_query_tokens"] = [hybrid_vi_tok(sq) for sq in e.get("sub_queries", [])]
else:
    print("[*] Không có enrichment — dùng original queries.")

# Extra dense vectors: sub-queries (không dùng HyDE)
extra_dense_items: list = []   # (qid, text)
if enrich_cache and USE_DECOMP:
    for it in test_data:
        qid  = it["id"]
        e    = enrich_cache.get(qid)
        orig = it["question"].strip()
        if not e:
            continue
        for sq in e.get("sub_queries", []):
            if sq.strip() and sq.strip() != orig:
                extra_dense_items.append((qid, sq))
        if USE_HYDE and e.get("hyde_clause"):
            extra_dense_items.append((qid, e["hyde_clause"]))

extra_dense_by_qid: dict = {}   # qid → list[list[int]] (article-level)
if extra_dense_items:
    print(f"[*] Encode {len(extra_dense_items)} enrichment texts...")
    extra_vecs = embedder.encode(
        [t for _, t in extra_dense_items],
        batch_size=DENSE_BATCH, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True,
    ).astype(np.float32)
    for (qid, _), vec in zip(extra_dense_items, extra_vecs):
        art_idxs = clause_dense_retrieve(vec, DENSE_K)
        extra_dense_by_qid.setdefault(qid, []).append(art_idxs)
    del extra_vecs; gc.collect(); torch.cuda.empty_cache()
    print(f"    extra dense cho {len(extra_dense_by_qid)} queries")

print("[*] Giải phóng embedder...")
del embedder; gc.collect(); torch.cuda.empty_cache()

[*] Load enrichment...
    2000 entries
[*] Encode 3030 enrichment texts...


Batches:   0%|          | 0/48 [00:00<?, ?it/s]

    extra dense cho 1610 queries
[*] Giải phóng embedder...


In [21]:
# Cell 9 — BM25 retrieval + clause-dense retrieval → RRF fusion

def _bm25_top(tokens: list) -> list:
    scores = bm25.get_scores(tokens)
    return np.argsort(-scores)[:BM25_K].tolist()


def rrf_fuse(rankings: list, k: int = RRF_K, top_n: int = CANDIDATE_K) -> list:
    score = {}; first = {}; order = 0
    for ranking in rankings:
        for rank, idx in enumerate(ranking, 1):
            order += 1
            first.setdefault(idx, order)
            score[idx] = score.get(idx, 0.0) + 1.0 / (k + rank)
    return [
        i for i, _ in
        sorted(score.items(), key=lambda kv: (-kv[1], first[kv[0]]))[:top_n]
    ]


print("[*] Retrieval (BM25 + clause-dense + enrichment) → RRF...")
candidates: dict = {}   # qid → list[int]

for item in tqdm(test_data, desc="Retrieve"):
    qid    = item["id"]
    q_toks = hybrid_vi_tok(item["question"])
    q_vec  = q_emb_by_id[qid]

    # BM25 legs
    bm25_main = _bm25_top(q_toks)
    bm25_extra: list = []
    e = enrich_cache.get(qid) if USE_ENRICHMENT else None
    if e:
        if USE_KEYWORDS and e.get("keywords_tokens"):
            bm25_extra.append(_bm25_top(e["keywords_tokens"]))
        if USE_DECOMP:
            for sq_toks in e.get("sub_query_tokens", []):
                if sq_toks:
                    bm25_extra.append(_bm25_top(sq_toks))

    # Dense leg (clause-level → article-level)
    dense_main = clause_dense_retrieve(q_vec, DENSE_K)

    all_rankings = [bm25_main, dense_main] + bm25_extra + extra_dense_by_qid.get(qid, [])
    candidates[qid] = rrf_fuse(all_rankings)

avg_pool = sum(len(v) for v in candidates.values()) / len(candidates)
print(f"    Done. Pool avg: {avg_pool:.1f}")

[*] Retrieval (BM25 + clause-dense + enrichment) → RRF...


Retrieve:   0%|          | 0/2000 [00:00<?, ?it/s]

    Done. Pool avg: 50.0


In [ ]:
# Cell 10 — Reranker: Sliding Window MaxP + Adaptive Soft-Floor Cutoff

CHUNK_SIZE    = 1200
CHUNK_OVERLAP = 200

print("[*] Load reranker (BGE-reranker-v2-m3)...")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=RERANK_DEVICE)
try:
    reranker.model.half()
except Exception:
    pass
torch.cuda.empty_cache()
print("    Reranker loaded.")


def adaptive_keep_soft(cand: list, scores: np.ndarray) -> list:
    if not cand:
        return []
    top         = scores[0]
    dynamic_gap = 0.2 if top >= 0.6 else 0.3
    soft_floor  = min(ABS_FLOOR, top - 0.1)
    thr         = max(soft_floor, top - dynamic_gap)
    keep        = [cand[i] for i in range(len(cand)) if scores[i] >= thr][:MAX_K]
    return keep or cand[:MIN_K]


def _sliding_chunks(text: str) -> list:
    if len(text) <= CHUNK_SIZE:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:min(start + CHUNK_SIZE, len(text))])
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks


def rerank_and_cut(query: str, pool: list) -> tuple:
    pairs: list = []
    pair_to_doc: list = []

    for c_idx, doc_idx in enumerate(pool):
        for chunk in _sliding_chunks(docs[doc_idx]):
            pairs.append([query, chunk])
            pair_to_doc.append(c_idx)

    if not pairs:
        return [], []

    batch_sz = RERANK_BATCH
    raw_scores: list = []
    i = 0
    while i < len(pairs):
        try:
            raw_scores.extend(
                reranker.predict(pairs[i:i+batch_sz], batch_size=batch_sz, show_progress_bar=False)
            )
            i += batch_sz
        except RuntimeError as ex:
            if "out of memory" in str(ex).lower() and batch_sz > 1:
                batch_sz = max(1, batch_sz // 2)
                torch.cuda.empty_cache()
            else:
                raise

    # MaxP per article: lấy max score trong tất cả chunks
    doc_scores = np.full(len(pool), -999.0, dtype=np.float32)
    for sc, c_idx in zip(raw_scores, pair_to_doc):
        if sc > doc_scores[c_idx]:
            doc_scores[c_idx] = sc

    order    = np.argsort(-doc_scores)
    cand_s   = [pool[i] for i in order]
    scores_s = doc_scores[order]
    final    = adaptive_keep_soft(cand_s, scores_s)

    relevant_articles = [metas[i] for i in final]
    seen, relevant_docs = set(), []
    for art in relevant_articles:
        parts = art.split("|")
        if len(parts) >= 2:
            doc_key = f"{parts[0]}|{parts[1]}"
            if doc_key not in seen:
                seen.add(doc_key)
                relevant_docs.append(doc_key)
    return relevant_docs, relevant_articles


print("[*] Rerank all queries...")
results   = []
processed = set()

if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, encoding="utf-8") as f:
        results = json.load(f)
    processed = {r["id"] for r in results}
    print(f"    Resume: {len(processed)} câu đã xong.")

for item in tqdm(test_data, desc="Rerank"):
    if item["id"] in processed:
        continue
    try:
        r_docs, r_arts = rerank_and_cut(item["question"], candidates[item["id"]])
    except Exception as ex:
        tqdm.write(f"  [!] Câu {item['id']}: {ex}")
        r_docs, r_arts = [], []
        torch.cuda.empty_cache(); gc.collect()

    results.append({
        "id":                item["id"],
        "question":          item["question"],
        "relevant_docs":     r_docs,
        "relevant_articles": r_arts,
    })

    if len(results) % 50 == 0:
        with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

print(f"    Done. {len(results)} câu.")


[*] Load reranker (BGE-reranker-v2-m3)...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

    Reranker loaded.
[*] Rerank all queries...


Rerank:   0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
# Cell 11 — Lưu kết quả + tạo submission.zip
import zipfile

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"[*] Saved {len(results)} results → {OUTPUT_PATH}")

n_arts = [len(r["relevant_articles"]) for r in results]
n_docs = [len(r["relevant_docs"])     for r in results]
print(f"    Avg articles/câu: {sum(n_arts)/len(n_arts):.2f}  (min={min(n_arts)}, max={max(n_arts)})")
print(f"    Avg docs/câu:     {sum(n_docs)/len(n_docs):.2f}  (min={min(n_docs)}, max={max(n_docs)})")

SUBMIT_PATH = "/kaggle/working/results.json"
submit_data = [
    {
        "id":                r["id"],
        "question":          r["question"],
        "answer":            "",
        "relevant_docs":     r["relevant_docs"],
        "relevant_articles": r["relevant_articles"],
    }
    for r in results
]
with open(SUBMIT_PATH, "w", encoding="utf-8") as f:
    json.dump(submit_data, f, ensure_ascii=False, indent=2)

ZIP_PATH = "/kaggle/working/submission.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(SUBMIT_PATH, "results.json")
print(f"[DONE] submission.zip → {ZIP_PATH}")